# Week 4 -- Strategy Reflection
### Imperial College -- Bayesian Optimisation Capstone

---

This notebook reflects on the shift to neural network surrogates and backpropagation-guided optimisation in Week 4, answering the seven module reflection questions. Written after submitting Week 4 queries and reviewing Week 3 results.

**Covers:**
- Support vectors and decision boundary points in function evaluations
- Using NN output gradients as directional signals
- Framing BBO as a classification task (good vs bad)
- Comparing surrogate model choices
- Steepest gradient dimensions and their implications
- How well the NN approximated the decision boundary
- NN vs simpler models: was the complexity worth it?

---
## Week 4 Pipeline Flowchart

```
╔══════════════════════════════════════════════════════════════════════════════╗
║              WEEK 4 NOTEBOOK PIPELINE  (per function, F1–F8)                ║
╚══════════════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 2 — Sync weekly_log → .npy                                        │
  │  Append W3 result to weekly_log list → idempotent save to disk          │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 3 — Data Inspection                                               │
  │  Load X, Y → sort descending by Y → print all observations              │
  │  → identify  best_X*  (highest observed Y)                              │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 4 — Gaussian Process Fit  (kept as fallback / comparison)         │
  │  Y_fit = log(|Y| + ε)   →   GP with RBF(ℓ=0.1, fixed), α=1e-10        │
  │  Output: gp object ready for UCB/EI scoring on random grid              │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 5 — Neural Network Surrogate  (TensorFlow / Assignment 15.1+15.2) │
  │                                                                          │
  │  Y_scaled = StandardScaler(log|Y|)  [F6: StandardScaler(Y) directly]   │
  │                                                                          │
  │  model = tf.keras.Sequential([          ← Assignment 15.1 style         │
  │    Dense(64, relu, L2=1e-3),                                             │
  │    Dropout(0.1),                                                         │
  │    Dense(64, relu, L2=1e-3),                                             │
  │    Dropout(0.1),                                                         │
  │    Dense(1)          ])                                                  │
  │                                                                          │
  │  @tf.function                           ← Assignment 15.2 style         │
  │  def train_step(x, y):                                                   │
  │    with tf.GradientTape() as tape:                                       │
  │      loss = MSE(y, model(x)) + L2_reg                                   │
  │    grads = tape.gradient(loss, model.trainable_variables)                │
  │    optimizer.apply_gradients(zip(grads, trainable_variables))            │
  │                                                                          │
  │  Train 1500 epochs, Adam lr=1e-3  [F8: 2000 epochs, hidden=128]        │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 6 — Dual Acquisition  (pick best candidate)                       │
  │                                                                          │
  │   STRATEGY A: NN Gradient Ascent     │  STRATEGY B: GP UCB / EI         │
  │   ─────────────────────────────────  │  ───────────────────────────     │
  │   x_opt = tf.Variable(best_X)        │  X_grid = random(30k, n_dims)   │
  │   opt = Adam(lr=5e-3)                │  score = μ + β·σ  (UCB)          │
  │                                      │  or EI formula  (F4 only)        │
  │   for step in range(800):            │  gp_candidate = argmax(score)    │
  │     with tf.GradientTape() as tape:  │                                  │
  │       score = model(x_opt)           │  [F4: 5 restarts from diverse    │
  │       loss  = −score                 │   seeds; F8: 3 restarts]         │
  │     g = tape.gradient(loss,[x_opt])  │                                  │
  │     opt.apply_gradients(g, x_opt)    │                                  │
  │     x_opt.assign(clip(x_opt,lo,hi))  │  trust-radius per function:      │
  │                                      │  F1=±0.25, F2/F3/F6=±0.20,      │
  │   grad_candidate = x_opt.numpy()     │  F5=±0.10, F7/F8=±0.15          │
  │                                      │                                  │
  │   ── compare NN-predicted score ──────────────────────────────────────  │
  │   next_x = whichever scores higher under the trained NN                 │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 7 — Input Gradient Sensitivity  (Assignment 15.2 style)           │
  │                                                                          │
  │  x_sens = tf.Variable(best_X)                                           │
  │  with tf.GradientTape() as tape:                                        │
  │    out = model(x_sens)                                                  │
  │  grads = |tape.gradient(out, x_sens)|                                   │
  │  → bar chart of |∂f/∂x_i|  → dimension importance ranking              │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 7b — SVM Classification Check  (secondary validation)             │
  │  Label top-30% = 'good' → train RBF-SVM → P(good | next_x)             │
  │  ✓ ≥ 0.5 endorsed  │  ~ 0.3–0.5 boundary  │  ✗ < 0.3 flagged          │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  CELL 8 — Summary + Portal String                                       │
  │  Print best_X*, best_Y*, next_x, portal_string  (6-decimal, hyphenated) │
  └───────────────────────────┬─────────────────────────────────────────────┘
                              ↓
              ┌───────────────────────────────────┐
              │  SUBMIT  portal_string  → Week 4  │
              │  Receive Y_new → add to weekly_log│
              │  for Week 5 notebook              │
              └───────────────────────────────────┘
```

**Key change from Week 3 → Week 4:** Cell 5 (GP-only) expanded to Cell 5 (NN surrogate, TF Sequential + GradientTape) + Cell 6 (gradient ascent on `tf.Variable` via `GradientTape`, replaces pure UCB grid search) + Cell 7 (input sensitivity via `GradientTape`, new). GP is kept as a fallback in Cell 6 only.

| Cell | Week 3 | Week 4 |
|------|--------|--------|
| 5 | GP fit only | GP fit (Cell 4) + **NN surrogate** (Cell 5, TF) |
| 6 | UCB/EI grid search | **Dual**: NN gradient ascent vs GP UCB/EI |
| 7 | Summary | **Input gradient sensitivity** (new) |
| 7b | — | **SVM classification check** (new) |
| 8/9 | Summary | Summary + Reflection |

---
## 1. Support Vectors and Decision Boundary Points

After three rounds of queries, certain observations sit near the edge between good and poor output regions -- these behave like support vectors in an SVM sense.

For **F5** (chemical process yield), the cluster of observations near [0.204, 0.878, 0.880, 0.871] is the clearest case. The initial data contains points with yield values ranging from roughly 200 to 900, while the three weekly queries pushed the value up to 1192. The boundary between the low-yield and high-yield regions runs somewhere through the 4D space, and the observations closest to that boundary -- those with yields around 800-900 -- act as support vectors. They define where the good region begins and point toward which direction to push next.

For **F7** (6D hyperparameter tuning), the three weekly queries show a consistent trajectory: 0.679, 0.775, 1.235. The observations around 0.7-0.8 straddle the boundary between average and good configurations. A trained SVM would place its margin near these points and indicate that moving from low x1, mid x2-x4, lower x6 toward slightly different x3 values crosses into the high-performance region.

For **F4** (warehouse placement), the situation is reversed. The good region defined by the Week 1 result (-24.548) is clearly separated from the poor region we got stuck in at -28.564. The boundary support vectors here warned us that the acquisition function was steering toward a worse attractor. Recognising boundary points earlier -- by labelling results as good/poor after Week 1 -- would have justified switching strategies sooner rather than waiting until Week 3.

The practical implication is straightforward: observations near the good/poor boundary provide the most information about where the optimum lies. Future queries should be placed near these boundary points to map the transition precisely, rather than randomly exploring the full space or exploiting only the proven peak.

---
## 2. Neural Network Output Gradients as Directional Signals

A Gaussian process models uncertainty but does not naturally produce input gradients that point toward better regions. A neural network surrogate does — through backpropagation, which TensorFlow's `tf.GradientTape` makes explicit.

**How it works in the notebooks (Assignment 15.2 pattern):**

```
# Cell 7 — sensitivity at best_X
x_sens = tf.Variable(best_X)          # declare input as a Variable (watched by tape)
with tf.GradientTape() as tape:
    out = model(x_sens)                # forward pass
grads = |tape.gradient(out, x_sens)|  # ∂output/∂x_i at best_X
```

The same `GradientTape` mechanism drives **Cell 6** gradient ascent — but there, `x_opt` (not model weights) is optimised: the tape watches `x_opt`, computes `loss = −model(x_opt)`, and `apply_gradients` shifts `x_opt` up the predicted surface. After 800 Adam steps the result is the local maximum of the NN's learned surface within the trust-region.

In **F7** (6D), the gradient magnitudes reveal that two or three dimensions drive most of the predicted output change, while the remaining dimensions have near-zero gradients at the current best. This means the NN effectively says: move along x3 and x5, but changes to x1 or x6 will not help much at this location.

In **F5** (unimodal 4D), if all four gradients are close to zero, the NN believes the current point is already near the peak — any direction decreases output. Non-zero gradients, by contrast, mean a better region exists in that direction. This is actionable: if |∂f/∂x_2| >> |∂f/∂x_1|, the next query should move along x_2.

In **F8** (8D), the gradient sensitivity ranking is the most valuable tool available. It compresses the 8D search: if five dimensions show low gradients and three show high gradients, the effective next-step search is 3D. `tf.GradientTape` makes this compression computationally free — one forward pass and one backward pass.

---
## 3. BBO as a Classification Task: Good vs Bad Outputs

A useful reframing is to label each observation as good (top 30% of observed outputs) or bad (bottom 70%), then train a classifier on the inputs. The decision boundary of that classifier traces the edge of the good region in input space.

**Logistic regression** would draw a linear boundary -- a hyperplane in the input space. For F1 (2D), where the good region appears to be a small interior patch surrounded by near-zero values, a linear boundary would be wrong because the good region is not separable by a straight line from any angle.

**SVMs with an RBF kernel** can draw curved, closed boundaries. This fits the expected structure of most functions here: a connected region of good inputs surrounded by poor ones. The SVM margin would place the boundary where the density of good vs bad examples balances. The trade-off is that with only 12-43 observations per function, the SVM might overfit -- a wide margin (high C penalty) would chase individual good observations rather than learning a general boundary.

**Neural networks** can approximate boundaries of arbitrary shape but are even more prone to overfitting at this scale. With 33 observations in F7 and 6 dimensions, a fully connected MLP risks memorising the training labels rather than learning the true boundary.

The key trade-off in all three cases is between misclassification and exploration. A conservative boundary (few false positives for good) keeps queries safe but may miss a good region that was unlucky in early samples. An aggressive boundary (high sensitivity) explores more freely but wastes queries on poor regions. For this project, a liberal SVM boundary -- one that errs toward classifying uncertain regions as potentially good -- is more appropriate, since the cost of a poor query (one wasted week) is less than the cost of never finding the optimum.

---
## 4. Model Choice: Interpretability vs Flexibility

**Linear regression** offers maximum interpretability — a coefficient per dimension shows directly how each input relates to the output. For F5 (unimodal), this could work near the peak where the relationship is approximately quadratic, but linear regression misses the curvature and would predict the wrong direction once queries leave the neighbourhood of the training data. Crucially, it provides no gradient that is useful for acquisition: the gradient is just the coefficient vector, which is constant everywhere.

**Gaussian processes with a fixed RBF kernel** sit in the middle. They interpolate smoothly between observations, provide uncertainty estimates (σ), and work well in low dimensions with small datasets. Their limitation here is the fixed kernel — same smoothness everywhere — and that they provide no differentiable surface suitable for `GradientTape`-based input optimisation.

**Neural networks (TF Sequential + GradientTape)** are the most flexible and are the primary surrogate in Week 4:

| Benefit | How it is used |
|---------|---------------|
| Gradients via `GradientTape` | Cell 6: gradient ascent on `tf.Variable(x_opt)` to find local max |
| No kernel assumptions | Learns asymmetric surfaces (F4's sharp landscape) |
| Scalable to 8D | F8 uses hidden=128, 2000 epochs without exponential cost |
| L2 regularisation via `kernel_regularizer` | Reduces overfitting on small datasets |

The costs are:
- No uncertainty estimates — cannot distinguish a confidently wrong prediction from a genuinely uncertain region (unlike GP's σ)
- Overfitting risk: solved here with `Dropout(0.1)` + `L2(1e-3)` in each Dense layer
- `@tf.function` decorator required on `train_step` to avoid Python overhead in the 1500-epoch loop

**Hybrid strategy used in Week 4:** the NN provides the gradient-guided candidate (Cell 6, Strategy A), the GP provides the uncertainty-guided candidate (Cell 6, Strategy B), and the one with the higher NN-predicted score wins. This captures the GP's ability to suggest unexplored regions while using the NN's learned surface as a consistent scoring oracle for both candidates.

---
## 5. Steepest Gradient Dimensions and Prioritisation

The Cell 7 sensitivity analysis in each notebook ranks input dimensions by |∂f/∂x_i| at the best known point. This ranking directly informs how to focus future experiments.

For **F8** (8D), with only 43 observations across 8 dimensions, the gradient ranking is the primary tool for reducing the effective search dimensionality. If dimensions x1, x4, and x7 show high gradients while x2, x3, x5, x6, x8 show low gradients at the current best, the practical next step is to vary x1, x4, and x7 while fixing the others near their best-known values. This turns an 8D problem into a 3D one for the next query.

For **F5** (4D, unimodal plateau), if all four gradients are near zero, it suggests the NN believes the current point is already close to the maximum and that marginal improvements require very precise perturbations. A near-zero gradient in all directions is also a signature of a local maximum in the NN's learned surface -- which may or may not correspond to the true maximum.

For **F4** (stuck), the gradient analysis at the Week 1 best point [0.889, 0.556, 0.778, 0.778] may reveal which dimension has the steepest gradient and therefore offers the most leverage for escape. This is more useful than random search because it gives a specific direction to try rather than hoping a random new point happens to be better.

Going into Week 5, the gradient rankings from Cell 7 across all eight functions provide a prioritised list of which dimensions to perturb for each function. This is a more principled form of the intuitive judgement calls made in Weeks 2 and 3.

---
## 6. Neural Network Decision Boundary Approximation

When the BBO problem is framed as binary classification (top 30% = good, rest = bad), the neural network learns a decision boundary in the input space. Backpropagation provides direct access to the gradient of the boundary's position with respect to each input dimension.

For **F2** (2D), the heatmap in Cell 8 of the function notebook makes the NN-learned boundary visible. The NN predicts high output in the high-x1, mid-x2 region, consistent with the observed best at [1.0, 0.326]. The boundary between the predicted-good and predicted-poor regions runs approximately diagonally across the 2D space. Compared to the GP posterior, the NN surface is sometimes more sharply peaked -- it is more willing to predict extreme values in unobserved regions because it has no uncertainty mechanism. This can be an advantage (more decisive gradient directions) or a disadvantage (overconfident recommendations).

For **F7** (6D), the decision boundary cannot be visualised directly, but the gradient of the NN output with respect to inputs at the boundary-crossing region provides the same information: which direction, in 6D, is 'more good' vs 'less good'. Backpropagation computes this gradient exactly at any input point, including near the current best, which is effectively a boundary point between explored and unexplored good territory.

The limitation is that the NN decision boundary is an artefact of the training data and architecture. With 33 observations in F7, the boundary will shift as new observations arrive. The boundary seen in Week 4 may not match the true function's structure -- it is a learned approximation that becomes more reliable as data accumulates.

---
## 7. Neural Network vs Simpler Models: Was the Complexity Worth It?

The honest answer depends on the function.

**For F1 and F2 (2D):** The GP is arguably sufficient. With only 2 input dimensions, the response surface is visualised directly in the Cell 8 heatmap, and the GP posterior already gives a clear picture. The NN adds `GradientTape`-based gradient computation (Cell 6, 7), but a GP with a differentiable kernel could also provide gradients. The main justification here is workflow uniformity — using the same TF Sequential + GradientTape structure across all 8 functions rather than a bespoke 2D approach.

**For F7 and F8 (6D, 8D):** The NN offers a clear advantage. The GP with a fixed RBF kernel treats all dimensions equally and makes no use of structured patterns. The NN learns asymmetric surfaces and — critically — provides the `GradientTape` sensitivity ranking that makes high-dimensional search tractable. For F8, the jump from 7.627 to 9.038 in Week 3 came from a very different region of the space; the NN can interpolate across that large jump better than a length-scale=0.1 RBF.

**Complexity added by TF vs PyTorch:**  
Both use the same MLP architecture (Sequential API). The key TF-specific patterns are:
- `@tf.function` on `train_step` — compiles the loop to a graph for ~3–5× faster training
- `tf.Variable(x_opt)` for gradient ascent input — automatically watched by `GradientTape` without `requires_grad_(True)`
- `x_opt.assign(tf.clip_by_value(...))` for trust-region — in-place update of the Variable
- `kernel_regularizer=l2(1e-3)` in each Dense layer — replaces PyTorch's `weight_decay` in the Adam constructor

**Quantitative comparison:** The NN training MSE (Cell 5) gives a rough fit quality indicator. If the final scaled MSE < 0.5, the NN explains most variance in the scaled outputs. For F7 and F8 (33–43 observations, 6–8 dimensions), achieving MSE < 0.3 suggests the NN captured the main structure rather than memorising noise.

**Conclusion:** The NN surrogate was worth the added complexity for F7 and F8 where `GradientTape` sensitivity analysis provides actionable dimension rankings. For F1 and F2, the main benefit was the consistent pipeline across all 8 functions — the same `train_step`, `GradientTape` ascent, and sensitivity cell structure, making the notebooks easier to compare and maintain.

---
*Reflection written after Week 4 submission | Imperial College Bayesian Optimisation Capstone*